# AeroFlow Exploratory Analysis

This notebook analyses AeroFlow's supply chain performance using the prepared datasets created during the data cleaning stage.

The analysis focuses on delivery reliability, supplier performance, inventory risk, demand and forecasting, and quality performance to identify operational risks and opportunities for improvement.

## 1. Import Libraries and Load Clean Data

In [ ]:
import pandas as pd

parts = pd.read_csv("../data/clean/parts_master_clean.csv")
purchase_orders = pd.read_csv("../data/clean/purchase_orders_clean.csv")
quality_incidents = pd.read_csv("../data/clean/quality_incidents_clean.csv")
supply_chain_history = pd.read_csv("../data/clean/supply_chain_history_clean.csv")

In [361]:
parts.shape, purchase_orders.shape, quality_incidents.shape, supply_chain_history.shape



((300, 9), (29666, 14), (368, 8), (280800, 13))

## 2. Overall Supply Chain Performance
### OTIF Performance


In [362]:
# Calculate overall OTIF rate
otif_pct = purchase_orders['otif_flag'].mean() * 100
round(otif_pct, 2)

np.float64(39.74)

AeroFlow achieved an overall OTIF rate of 39.74%, meaning fewer than half of purchase orders were delivered both on time and in full. This indicates a significant opportunity to improve supplier delivery reliability.

In [363]:
# Calculate on-time rate
on_time_pct = purchase_orders['on_time_flag'].mean() * 100
round(on_time_pct, 2)


np.float64(44.15)

AeroFlow achieved an overall on-time delivery rate of 44.15%. This is slightly higher than the OTIF rate of 39.74%, indicating that some orders arrived on time but did not meet the full quantity requirement.

In [364]:
# Calculate in-full rate
in_full_pct = purchase_orders['in_full_flag'].mean() * 100
round(in_full_pct, 2)


np.float64(88.69)

AeroFlow achieved an in-full delivery rate of 88.69%, which is substantially stronger than its on-time rate of 44.15%. This suggests delivery delays are the main factor reducing overall OTIF performance.

In [365]:
# Calculate average delivery variance
avg_delivery_variance_days = purchase_orders["delivery_variance_days"].mean()
round(avg_delivery_variance_days, 2)                                                                                                                                                                                  

np.float64(1.18)

AeroFlow deliveries arrived an average of 1.18 days later than the promised delivery date. Combined with the low on-time rate of 44.15%, this confirms that delivery timeliness is a key operational weakness.


In [366]:
# Calculate stockout rate
stockout_rate = supply_chain_history['stockout_flag'].mean() * 100
round(stockout_rate, 2)


np.float64(0.57)

AeroFlow recorded a stockout rate of 0.57%, meaning stockouts occurred in a small proportion of weekly part-site records. Although the overall rate is low, later analysis should identify whether stockouts are concentrated among critical parts, specific sites or particular suppliers.

In [367]:
# Calculate backorder rate
backorder_rate = supply_chain_history['backorder_flag'].mean() * 100
round(backorder_rate, 2)

np.float64(0.83)

AeroFlow recorded a backorder rate of 0.83%. Although this is low overall, the next stage should examine whether backorders are concentrated among specific parts, sites or suppliers.

## 3. Supplier Performance

In [368]:
supplier_summary = (
    purchase_orders
    .groupby("supplier_id")
    .agg(
        order_count=("po_id", "count"),
        on_time_rate=("on_time_flag", "mean"),
        in_full_rate=("in_full_flag", "mean"),
        otif_rate=("otif_flag", "mean"),
        avg_delivery_delay_days=("delivery_variance_days", "mean"),
    )
    .reset_index()
)

supplier_summary.head(10)

,supplier_id,order_count,on_time_rate,in_full_rate,otif_rate,avg_delivery_delay_days
0,SUP001,659,0.578149,0.918058,0.520486,0.267071
1,SUP002,763,0.281782,0.871560,0.256881,2.245085
2,SUP003,633,0.330174,0.845182,0.281201,1.914692
3,SUP004,662,0.564955,0.922961,0.525680,0.256798
4,SUP005,436,0.337156,0.848624,0.282110,1.931193
5,SUP006,1096,0.313869,0.849453,0.263686,2.114051
6,SUP007,831,0.531889,0.925391,0.489771,0.393502
7,SUP008,587,0.560477,0.916525,0.514480,0.183986
8,SUP009,649,0.302003,0.852080,0.252696,1.927581
9,SUP010,930,0.567742,0.917204,0.519355,0.352688


SUP033 is the weakest performing supplier, with an OTIF rate of 3.01% across 631 purchase orders. Its on-time rate is only 3.65%, while its in-full rate is 76.23%, indicating that chronic delivery lateness is the primary driver of poor OTIF performance.

In [369]:
parts.loc[
    parts["supplier_id_primary"] == "SUP033", ["part_id", "criticality_class", "supplier_risk_class"]]


,part_id,criticality_class,supplier_risk_class
60,P00061,C,High
126,P00127,C,High
128,P00129,C,High
163,P00164,C,High
255,P00256,B,High
263,P00264,A,High
293,P00294,B,High


SUP033 primarily supplies seven AeroFlow parts, all of which are classified as High supplier risk. The portfolio includes one Class A and two Class B criticality parts, increasing the operational significance of SUP033's poor delivery performance.

In [370]:
sup033_avg_delivery_variance_days = (
purchase_orders.loc[
    purchase_orders["supplier_id"] == "SUP033",
    "delivery_variance_days"
]
    .mean()
)

round(sup033_avg_delivery_variance_days, 2)

np.float64(6.09)

SUP033 deliveries arrived an average of 6.09 days later than promised, compared with AeroFlow's overall average delay of 1.18 days. This reinforces that SUP033's poor OTIF performance is primarily driven by persistent and significant delivery lateness.

In [371]:
supplier_risk_summary = purchase_orders.merge(
    parts,
    on="part_id",
    how="left"
)
supplier_risk_summary.shape

supplier_risk_summary["supplier_risk_class"].value_counts()

supplier_risk_class
Low       15408
Medium    13627
High        631
Name: count, dtype: int64

In [372]:
risk_otif = (
    supplier_risk_summary
    .groupby("supplier_risk_class")["otif_flag"]
    .mean()
    * 100
)

risk_otif

supplier_risk_class
High       3.011094
Low       52.518172
Medium    26.990533
Name: otif_flag, dtype: float64

OTIF performance declines sharply as supplier risk increases. Low-risk suppliers achieved 52.52% OTIF, compared with 27.00% for medium-risk suppliers and 3.01% for the high-risk category. However, the high-risk result is entirely driven by SUP033, so it should be treated as evidence of a major supplier specific risk rather than a broad conclusion about all high-risk suppliers.

In [373]:
supplier_avg_delivery_delay = (
    purchase_orders
    .groupby("supplier_id")["delivery_variance_days"]
    .mean().sort_values(ascending=False)
.reset_index())
supplier_avg_delivery_delay.head(10)

,supplier_id,delivery_variance_days
0,SUP033,6.088748
1,SUP023,2.367857
2,SUP002,2.245085
3,SUP015,2.205811
4,SUP039,2.179739
5,SUP026,2.158824
6,SUP006,2.114051
7,SUP014,2.081851
8,SUP021,2.078394
9,SUP025,2.068140


SUP033 recorded the highest average delivery delay at 6.09 days late, more than double the next-worst supplier. This confirms that SUP033 is a major delivery reliability outlier and should be treated as a priority supplier risk.

In [374]:
quality_performance = (
    quality_incidents
    .groupby("supplier_id")
    .agg(incident_count=("incident_id", "count"))
    .reset_index()
    .sort_values("incident_count", ascending=False)
)

quality_performance.head(10)

quality_performance[
    quality_performance["supplier_id"] == "SUP033"
]

,supplier_id,incident_count
32,SUP033,11


SUP033 recorded 11 quality incidents, which is not among the highest supplier incident counts. This suggests that its primary performance issue is delivery reliability rather than quality frequency.

In [375]:
supplier_order_count_clean = supplier_summary[
    ["supplier_id", "order_count"]
].copy()

In [376]:
supplier_quality_rate = supplier_order_count_clean.merge(
    quality_performance,
    on="supplier_id",
    how="left"
)

supplier_quality_rate["incident_count"] = (
    supplier_quality_rate["incident_count"].fillna(0)
)

supplier_quality_rate.head()

,supplier_id,order_count,incident_count
0,SUP001,659,3
1,SUP002,763,10
2,SUP003,633,6
3,SUP004,662,6
4,SUP005,436,1


In [377]:
supplier_quality_rate["quality_incidents_per_100_orders"] = (
    supplier_quality_rate["incident_count"]
    / supplier_quality_rate["order_count"]
    * 100
)

supplier_quality_rate = (
    supplier_quality_rate
    .sort_values(
        "quality_incidents_per_100_orders",
        ascending=False
    )
    .reset_index(drop=True)
)

supplier_quality_rate.head(10)


,supplier_id,order_count,incident_count,quality_incidents_per_100_orders
0,SUP023,280,7,2.500000
1,SUP024,821,16,1.948843
2,SUP027,1311,24,1.830664
3,SUP018,607,11,1.812191
4,SUP040,1215,22,1.810700
5,SUP033,631,11,1.743265
6,SUP034,578,10,1.730104
7,SUP009,649,11,1.694915
8,SUP013,1156,19,1.643599
9,SUP006,1096,18,1.642336


When adjusted for purchase-order volume, SUP033 recorded 1.74 quality incidents per 100 orders, ranking sixth highest among suppliers. This indicates some quality exposure, but delivery reliability remains the more severe issue for SUP033.

Quality incident dates extend to February 2025, while purchase-order dates end in December 2024. Incidents per 100 orders is therefore a whole-file comparison, not a matched-period defect rate.

In [378]:
quality_incidents["defect_severity"].value_counts()



defect_severity
Minor       243
Major        91
Critical     34
Name: count, dtype: int64

In [379]:
serious_incidents = quality_incidents[
    quality_incidents["defect_severity"].isin(["Major", "Critical"])
]


SUP033 recorded seven Major or Critical quality incidents, placing it among the suppliers with the highest number of serious quality issues. Although its total incident count was not the highest, the severity profile adds further concern alongside its poor delivery performance.

In [380]:
serious_incident_count = (
    serious_incidents
    .groupby("supplier_id")
    .agg(serious_incident_count=("incident_id", "count"))
    .reset_index()
)

serious_incident_rate = supplier_order_count_clean.merge(
    serious_incident_count,
    on="supplier_id",
    how="left"
)

serious_incident_rate.head()

,supplier_id,order_count,serious_incident_count
0,SUP001,659,1.0
1,SUP002,763,2.0
2,SUP003,633,4.0
3,SUP004,662,NaN
4,SUP005,436,NaN


In [381]:
serious_incident_rate["serious_incident_count"] = (
    serious_incident_rate["serious_incident_count"].fillna(0)
)

serious_incident_rate["serious_incidents_per_100_orders"] = (
    serious_incident_rate["serious_incident_count"]
    / serious_incident_rate["order_count"]
    * 100
)

serious_incident_rate = (
    serious_incident_rate
    .sort_values(
        "serious_incidents_per_100_orders",
        ascending=False
    )
    .reset_index(drop=True)
)

serious_incident_rate.head(10)

,supplier_id,order_count,serious_incident_count,serious_incidents_per_100_orders
0,SUP033,631,7.0,1.109350
1,SUP016,385,4.0,1.038961
2,SUP025,543,5.0,0.920810
3,SUP034,578,5.0,0.865052
4,SUP030,1171,9.0,0.768574
5,SUP038,967,7.0,0.723888
6,SUP019,691,5.0,0.723589
7,SUP037,842,6.0,0.712589
8,SUP027,1311,9.0,0.686499
9,SUP008,587,4.0,0.681431


When adjusted for purchase-order volume, SUP033 recorded the highest rate of Major or Critical quality incidents at 1.11 per 100 orders. Combined with its 3.01% OTIF rate, 6.09-day average delivery delay and High supplier-risk classification, SUP033 represents AeroFlow's most significant supplier performance risk.

## 4. Inventory and Parts Risk

This section examines stockout and backorder patterns across AeroFlow's parts portfolio to identify where inventory availability may create operational risk.

The analysis focuses on the frequency of stockouts and backorders, the criticality of affected parts, and whether inventory risk is concentrated in specific parts or sites.

### Stockout Frequency by Part

In [382]:
part_stockouts = (
    supply_chain_history
    .groupby("part_id")
    .agg(
        stockout_count=("stockout_flag", "sum")
    )
    .reset_index()
)

part_stockouts = (
    part_stockouts
    .sort_values("stockout_count", ascending=False)
    .reset_index(drop=True)
)

part_stockouts.head(10)

,part_id,stockout_count
0,P00179,101
1,P00124,71
2,P00043,64
3,P00062,52
4,P00288,49
5,P00105,48
6,P00087,48
7,P00165,45
8,P00197,42
9,P00146,42


In [383]:
part_stockout_risk = part_stockouts.merge(
    parts[
        ["part_id", "part_family", "criticality_class", "supplier_risk_class"]
    ],
    on="part_id",
    how="left"
)

part_stockout_risk.head(10)

,part_id,stockout_count,part_family,criticality_class,supplier_risk_class
0,P00179,101,Electrical,C,Low
1,P00124,71,Electrical,C,Low
2,P00043,64,Electrical,C,Low
3,P00062,52,Electrical,A,Low
4,P00288,49,Electrical,C,Low
5,P00105,48,Structure,C,Low
6,P00087,48,Fasteners,B,Low
7,P00165,45,Electrical,C,Low
8,P00197,42,Electrical,C,Low
9,P00146,42,Electrical,B,Low


The highest stockout counts are concentrated largely within Electrical parts. Most of the top-stockout parts are Class C, but the list also includes higher-criticality items, including Class A part P00062 with 52 stockout records and several Class B parts. Despite the stockout frequency, these parts are associated with Low supplier-risk classifications, suggesting the inventory issue may be driven by demand, stocking policy or replenishment rather than supplier risk alone.

In [384]:
criticality_stockouts = (
    part_stockout_risk
    .groupby("criticality_class")
    .agg(
        stockout_count=("stockout_count", "sum")
    )
    .sort_values("stockout_count", ascending=False)
    .reset_index()
)

criticality_stockouts

,criticality_class,stockout_count
0,C,1140
1,B,314
2,A,160


In [385]:
parts_by_criticality = (
    parts
    .groupby("criticality_class")
    .agg(
        part_count=("part_id", "count")
    )
    .reset_index()
)

criticality_stockout_rate = parts_by_criticality.merge(
    criticality_stockouts,
    on="criticality_class",
    how="left"
)

criticality_stockout_rate["stockouts_per_part"] = (
    criticality_stockout_rate["stockout_count"]
    / criticality_stockout_rate["part_count"]
)

criticality_stockout_rate.sort_values(
    "stockouts_per_part",
    ascending=False
)

,criticality_class,part_count,stockout_count,stockouts_per_part
2,C,157,1140,7.261146
0,A,48,160,3.333333
1,B,95,314,3.305263


After adjusting for the number of parts in each criticality class, Class C parts still recorded the highest stockout frequency at 7.26 stockout records per part. Class A and B parts were much lower at 3.33 and 3.31 respectively. This suggests stockout risk is concentrated more heavily among lower-criticality Class C parts, although the presence of Class A stockouts still creates operational risk.

In [386]:
part_family_stockouts = (
    part_stockout_risk
    .groupby("part_family")
    .agg(
        stockout_count=("stockout_count", "sum")
    )
    .sort_values("stockout_count", ascending=False)
    .reset_index()
)

part_family_stockouts.head(10)

,part_family,stockout_count
0,Electrical,655
1,Fasteners,266
2,LandingGear,216
3,Structure,166
4,Engine,134
5,Hydraulics,100
6,Avionics,74
7,Cabin,3


In [387]:
parts_per_family = (
    parts
    .groupby("part_family")
    .agg(
        part_count=("part_id", "count")
    )
    .reset_index()
)

part_family_stockout_rate = parts_per_family.merge(
    part_family_stockouts,
    on="part_family",
    how="left"
)

part_family_stockout_rate["stockouts_per_part"] = (
    part_family_stockout_rate["stockout_count"]
    / part_family_stockout_rate["part_count"]
)

part_family_stockout_rate.sort_values(
    "stockouts_per_part",
    ascending=False
)

,part_family,part_count,stockout_count,stockouts_per_part
2,Electrical,41,655,15.975610
6,LandingGear,27,216,8.000000
7,Structure,36,166,4.611111
4,Fasteners,61,266,4.360656
3,Engine,32,134,4.187500
5,Hydraulics,33,100,3.030303
0,Avionics,37,74,2.000000
1,Cabin,33,3,0.090909


Electrical parts recorded the highest stockout frequency at 15.98 stockouts per part, nearly double the rate for LandingGear at 8.00. This confirms that the high Electrical stockout count is not simply due to a larger number of parts and indicates a concentrated inventory availability issue within this part family.

In [388]:
site_stockouts = (
    supply_chain_history
    .groupby("site_id")
    .agg(
        stockout_count=("stockout_flag", "sum")
    )
    .sort_values("stockout_count", ascending=False)
    .reset_index()
)

site_stockouts

,site_id,stockout_count
0,SITE02,331
1,SITE03,303
2,SITE05,290
3,SITE01,282
4,SITE06,212
5,SITE04,196


Stockouts were distributed across all six AeroFlow sites, with SITE02 recording the highest count at 331, followed by SITE03 at 303. The relatively narrow spread between sites suggests inventory availability is a broader network issue rather than being isolated to a single location.

In [389]:
site_part_stockouts = (
    supply_chain_history
    .groupby(["site_id", "part_id"])
    .agg(
        stockout_count=("stockout_flag", "sum")
    )
    .reset_index()
)

site_part_stockouts.sort_values(
    "stockout_count",
    ascending=False
).head(10)


,site_id,part_id,stockout_count
361,SITE02,P00062,25
778,SITE03,P00179,22
342,SITE02,P00043,20
478,SITE02,P00179,17
1364,SITE05,P00165,17
123,SITE01,P00124,17
1378,SITE05,P00179,16
178,SITE01,P00179,16
1286,SITE05,P00087,16
1323,SITE05,P00124,16


In [390]:
site_part_risk = site_part_stockouts.merge(
    parts[
        ["part_id", "part_family", "criticality_class"]
    ],
    on="part_id",
    how="left"
)

site_part_risk.sort_values(
    "stockout_count",
    ascending=False
).head(10)

,site_id,part_id,stockout_count,part_family,criticality_class
361,SITE02,P00062,25,Electrical,A
778,SITE03,P00179,22,Electrical,C
342,SITE02,P00043,20,Electrical,C
478,SITE02,P00179,17,Electrical,C
1364,SITE05,P00165,17,Electrical,C
123,SITE01,P00124,17,Electrical,C
1378,SITE05,P00179,16,Electrical,C
178,SITE01,P00179,16,Electrical,C
1286,SITE05,P00087,16,Fasteners,B
1323,SITE05,P00124,16,Electrical,C


The most severe site-part stockout combination was P00062 at SITE02, with 25 stockout records. P00062 is an Electrical Class A part, making this a higher-priority inventory risk than the more frequent Class C stockouts elsewhere. Electrical parts dominate the highest-risk site-part combinations, reinforcing the earlier finding that this part family has the greatest stockout exposure.

In [391]:
part_backorders = (
    supply_chain_history
    .groupby("part_id")
    .agg(
        backorder_count=("backorder_flag", "sum")
    )
    .sort_values("backorder_count", ascending=False)
    .reset_index()
)

part_backorders.head(10)

,part_id,backorder_count
0,P00179,128
1,P00124,105
2,P00165,85
3,P00146,84
4,P00016,77
5,P00105,69
6,P00043,67
7,P00288,67
8,P00197,57
9,P00087,51


Backorders are concentrated among many of the same parts that also experience frequent stockouts. P00179 and P00124 are the strongest examples, recording both high stockout and high backorder counts. This indicates persistent inventory pressure on a subset of parts rather than isolated one-off availability issues.

In [392]:
part_inventory_risk = part_stockouts.merge(
    part_backorders,
    on="part_id",
    how="outer"
)

part_inventory_risk = part_inventory_risk.merge(
    parts[
        ["part_id", "part_family", "criticality_class", "supplier_risk_class"]
    ],
    on="part_id",
    how="left"
)

part_inventory_risk["inventory_risk_flag_count"] = (
    part_inventory_risk["stockout_count"]
    + part_inventory_risk["backorder_count"]
)

part_inventory_risk.sort_values(
    "inventory_risk_flag_count",
    ascending=False
).head(10)

,part_id,stockout_count,backorder_count,part_family,criticality_class,supplier_risk_class,inventory_risk_flag_count
178,P00179,101,128,Electrical,C,Low,229
123,P00124,71,105,Electrical,C,Low,176
42,P00043,64,67,Electrical,C,Low,131
164,P00165,45,85,Electrical,C,Low,130
145,P00146,42,84,Electrical,B,Low,126
104,P00105,48,69,Structure,C,Low,117
15,P00016,40,77,LandingGear,B,Low,117
287,P00288,49,67,Electrical,C,Low,116
196,P00197,42,57,Electrical,C,Low,99
86,P00087,48,51,Fasteners,B,Low,99


The combined inventory-risk view shows that a small group of parts account for a disproportionate number of stockout and backorder flags. P00179 is the highest-risk part with 229 combined flag counts, followed by P00124 with 176. Electrical parts dominate the highest-risk group, while supplier-risk classifications remain Low, suggesting that inventory policy, replenishment or demand variability may be more important drivers than supplier risk.

The combined count adds stockout and backorder flags; a weekly part-site record contributes twice when both flags are present.

In [393]:
site_backorders = (
    supply_chain_history
    .groupby("site_id")
    .agg(
        backorder_count=("backorder_flag", "sum")
    )
    .sort_values("backorder_count", ascending=False)
    .reset_index()
)

site_backorders

,site_id,backorder_count
0,SITE02,430
1,SITE01,412
2,SITE05,390
3,SITE06,380
4,SITE03,373
5,SITE04,349


Backorders were recorded across all six AeroFlow sites, with SITE02 showing the highest count at 430. However, SITE01, SITE05 and SITE06 were also relatively high, indicating that backorder pressure is distributed across the network rather than being isolated to one site.

In [394]:
site_inventory_risk = site_stockouts.merge(
    site_backorders,
    on="site_id",
    how="outer"
)

site_inventory_risk["inventory_risk_flag_count"] = (
    site_inventory_risk["stockout_count"]
    + site_inventory_risk["backorder_count"]
)

site_inventory_risk.sort_values(
    "inventory_risk_flag_count",
    ascending=False
)

,site_id,stockout_count,backorder_count,inventory_risk_flag_count
1,SITE02,331,430,761
0,SITE01,282,412,694
4,SITE05,290,390,680
2,SITE03,303,373,676
5,SITE06,212,380,592
3,SITE04,196,349,545


SITE02 recorded the highest combined inventory-risk exposure, with 761 stockout and backorder flags. However, SITE01, SITE05 and SITE03 also recorded relatively high totals, indicating that inventory availability issues affect the wider AeroFlow network rather than a single site.

## 5. Demand and Forecasting

This section compares forecast demand with actual consumption to assess forecast accuracy, identify over- and under-forecasting patterns, and understand whether forecast performance varies by part, site or maintenance activity.

### Forecast Error

In [395]:

supply_chain_history["forecast_error"] = (
    supply_chain_history["forecast_qty"]
    - supply_chain_history["consumption_qty"]
)

avg_forecast_error = supply_chain_history["forecast_error"].mean()
round(avg_forecast_error, 2)

np.float64(0.05)

Average forecast error was +0.05 units, indicating that forecasts were almost unbiased overall. However, this average may mask larger positive and negative errors that offset each other, so additional accuracy measures are required.

In [396]:
supply_chain_history["absolute_forecast_error"] = (
    supply_chain_history["forecast_error"].abs()
)

mae = supply_chain_history["absolute_forecast_error"].mean()
round(mae, 2)

np.float64(1.45)

Mean Absolute Error (MAE) was 1.45 units, meaning AeroFlow forecasts differed from actual consumption by an average of 1.45 units per weekly part-site record. This shows that the near-zero average forecast error was driven by over- and under-forecasting offsetting each other rather than perfect forecast accuracy.

In [397]:
forecast_direction = (
    supply_chain_history["forecast_error"]
    .apply(
        lambda x: "Over Forecast"
        if x > 0
        else "Under Forecast"
        if x < 0
        else "Exact"
    )
)

forecast_direction.value_counts()

forecast_error
Over Forecast     120689
Under Forecast     83024
Exact              77087
Name: count, dtype: int64

Forecasts were more frequently above actual consumption than below it. Over-forecasting occurred in 120,689 records, compared with 83,024 under-forecast records and 77,087 exact matches. This indicates a slight overall tendency toward over-forecasting.

In [398]:
maintenance_forecast_accuracy = (
    supply_chain_history
    .groupby("planned_maintenance")
    .agg(
        mae=("absolute_forecast_error", "mean")
    )
    .reset_index()
)

maintenance_forecast_accuracy

,planned_maintenance,mae
0,False,1.717936
1,True,1.089241


Forecast accuracy was stronger during planned maintenance periods. Mean Absolute Error was 1.09 units when planned maintenance was scheduled, compared with 1.72 units when no planned maintenance was recorded. This suggests maintenance-related demand may be more predictable than routine demand.

In [399]:
forecast_type_accuracy = (
    supply_chain_history
    .groupby("forecast_type")
    .agg(
        mae=("absolute_forecast_error", "mean")
    )
    .reset_index()
)

forecast_type_accuracy

,forecast_type,mae
0,Adjusted,1.34590
1,Baseline,1.46879


Adjusted forecasts achieved a lower Mean Absolute Error of 1.35 units compared with 1.47 units for baseline forecasts. This suggests forecast adjustments improve accuracy, although the improvement is relatively modest.

In [400]:
forecast_accuracy_by_family = (
    supply_chain_history
    .merge(
        parts[["part_id", "part_family"]],
        on="part_id",
        how="left"
    )
    .groupby("part_family")
    .agg(
        mae=("absolute_forecast_error", "mean")
    )
    .sort_values("mae", ascending=False)
    .reset_index()
)

forecast_accuracy_by_family

,part_family,mae
0,Cabin,2.486435
1,Avionics,1.681306
2,Fasteners,1.546553
3,Structure,1.381203
4,LandingGear,1.340891
5,Electrical,1.125052
6,Hydraulics,1.071096
7,Engine,0.861879


Forecast accuracy varies significantly by part family. Cabin parts recorded the highest MAE at 2.49 units, making them the least predictable family, while Engine parts had the lowest MAE at 0.86 units. This suggests forecast improvement efforts should be targeted by part family rather than applied uniformly across the portfolio.

In [401]:
family_forecast_inventory = forecast_accuracy_by_family.merge(
    part_family_stockout_rate,
    on="part_family",
    how="left"
)

family_forecast_inventory = (
    family_forecast_inventory
    .sort_values("mae", ascending=False)
    .reset_index(drop=True)
)

family_forecast_inventory[
    ["part_family", "mae", "stockouts_per_part"]
]


,part_family,mae,stockouts_per_part
0,Cabin,2.486435,0.090909
1,Avionics,1.681306,2.000000
2,Fasteners,1.546553,4.360656
3,Structure,1.381203,4.611111
4,LandingGear,1.340891,8.000000
5,Electrical,1.125052,15.975610
6,Hydraulics,1.071096,3.030303
7,Engine,0.861879,4.187500


Part-family analysis does not show a clear relationship between forecast accuracy and stockout exposure. Cabin parts had the highest forecast error but almost no stockouts, while Electrical parts had relatively low forecast error but the highest stockout rate at 15.98 per part. This suggests that stockout risk is more likely being driven by replenishment, safety-stock policy or supply constraints than forecast accuracy alone.

In [402]:
forecast_accuracy_by_site = (
    supply_chain_history
    .groupby("site_id")
    .agg(
        mae=("absolute_forecast_error", "mean")
    )
    .sort_values("mae", ascending=False)
    .reset_index()
)

forecast_accuracy_by_site

,site_id,mae
0,SITE06,2.111902
1,SITE04,2.079145
2,SITE01,1.137970
3,SITE03,1.121218
4,SITE05,1.112714
5,SITE02,1.108675


Forecast accuracy varies materially by site. SITE06 and SITE04 recorded the highest Mean Absolute Error at 2.11 and 2.08 units respectively, while the remaining four sites were clustered close to 1.1 units. This suggests forecast performance issues are concentrated primarily at SITE06 and SITE04 rather than being consistent across the network.

In [403]:
site_forecast_inventory = forecast_accuracy_by_site.merge(
    site_inventory_risk,
    on="site_id",
    how="left"
)
site_forecast_inventory[
    ["site_id", "mae", "inventory_risk_flag_count"]
]

,site_id,mae,inventory_risk_flag_count
0,SITE06,2.111902,592
1,SITE04,2.079145,545
2,SITE01,1.137970,694
3,SITE03,1.121218,676
4,SITE05,1.112714,680
5,SITE02,1.108675,761


Site-level analysis also shows no clear relationship between forecast accuracy and inventory-risk exposure. SITE06 and SITE04 had the highest forecast error but relatively lower inventory-risk totals, while SITE02 had the lowest MAE yet the highest combined stockout and backorder count. This suggests inventory availability issues are likely being driven more by replenishment, stocking policy or supply constraints than forecasting error alone.

In [404]:
forecast_accuracy_by_part = (
    supply_chain_history
    .groupby("part_id")
    .agg(
        mae=("absolute_forecast_error", "mean")
    )
    .sort_values("mae", ascending=False)
    .reset_index()
)

forecast_accuracy_by_part.head(10)

,part_id,mae
0,P00101,3.458333
1,P00257,3.407051
2,P00005,3.326923
3,P00108,3.216880
4,P00002,3.201923
5,P00060,3.130342
6,P00231,3.054487
7,P00238,3.050214
8,P00267,2.858974
9,P00018,2.816239


The highest forecast errors are concentrated among a small group of individual parts. P00101 recorded the highest MAE at 3.46 units, followed by P00257 at 3.41 and P00005 at 3.33. These parts should be prioritised for deeper investigation into demand variability and forecast assumptions.

In [405]:
forecast_part_risk = forecast_accuracy_by_part.merge(
    parts[
        ["part_id", "part_family", "criticality_class"]
    ],
    on="part_id",
    how="left"
)

forecast_part_risk.head(10)

,part_id,mae,part_family,criticality_class
0,P00101,3.458333,Cabin,C
1,P00257,3.407051,Cabin,C
2,P00005,3.326923,Cabin,C
3,P00108,3.216880,Cabin,C
4,P00002,3.201923,Cabin,C
5,P00060,3.130342,Cabin,C
6,P00231,3.054487,Cabin,C
7,P00238,3.050214,Cabin,C
8,P00267,2.858974,Cabin,C
9,P00018,2.816239,Cabin,C


The hardest-to-forecast individual parts are all Cabin Class C items. This confirms that forecast error is concentrated within lower-criticality Cabin demand rather than higher-criticality parts. While this represents a forecasting improvement opportunity, it is less operationally urgent than the Electrical stockout risk identified earlier.

## 6. Quality Performance

This section examines defect patterns and scrap impact to identify where quality issues are concentrated across suppliers, part families and criticality classes.

### Defect Type Distribution

In [406]:
defect_type_summary = (
    quality_incidents
    .groupby("defect_type")
    .agg(
        incident_count=("incident_id", "count")
    )
    .sort_values("incident_count", ascending=False)
    .reset_index()
)

defect_type_summary

,defect_type,incident_count
0,Material,67
1,Packaging,66
2,Surface finish,66
3,Documentation,63
4,Certification,58
5,Dimensional,48


Defect incidents were relatively evenly distributed across categories. Material defects were the most common at 67 incidents, closely followed by Packaging and Surface Finish at 66 each. No single defect type dominates, suggesting quality issues are spread across multiple failure modes rather than being driven by one recurring problem.

In [407]:
defect_scrap_summary = (
    quality_incidents
    .groupby("defect_type")
    .agg(
        total_scrap_qty=("scrap_qty", "sum")
    )
    .sort_values("total_scrap_qty", ascending=False)
    .reset_index()
)

defect_scrap_summary

,defect_type,total_scrap_qty
0,Documentation,96
1,Material,82
2,Packaging,71
3,Certification,71
4,Surface finish,70
5,Dimensional,66


Documentation defects generated the highest total scrap quantity at 96 units, despite not being the most frequent defect type. Material defects were second at 82 units. This shows that incident frequency alone does not fully represent quality impact, and scrap quantity provides an additional measure of severity.

In [408]:
quality_by_family = (
    quality_incidents
    .merge(
        parts[["part_id", "part_family", "criticality_class"]],
        on="part_id",
        how="left"
    )
    .groupby("part_family")
    .agg(
        incident_count=("incident_id", "count"),
        total_scrap_qty=("scrap_qty", "sum")
    )
    .sort_values("incident_count", ascending=False)
    .reset_index()
)

quality_by_family

,part_family,incident_count,total_scrap_qty
0,Fasteners,77,86
1,Cabin,63,105
2,Avionics,51,73
3,Electrical,45,45
4,Hydraulics,37,39
5,Structure,36,40
6,LandingGear,35,43
7,Engine,24,25


Fasteners recorded the highest number of quality incidents at 77, while Cabin generated the highest scrap quantity at 105 units. This indicates that the most frequent quality issues are not necessarily the most operationally costly, and both incident frequency and scrap impact should be considered when prioritising quality improvement.

In [409]:
quality_by_criticality = (
    quality_incidents
    .merge(
        parts[["part_id", "criticality_class"]],
        on="part_id",
        how="left"
    )
    .groupby("criticality_class")
    .agg(
        incident_count=("incident_id", "count"),
        total_scrap_qty=("scrap_qty", "sum")
    )
    .sort_values("incident_count", ascending=False)
    .reset_index()
)

quality_by_criticality

,criticality_class,incident_count,total_scrap_qty
0,C,193,238
1,B,139,180
2,A,36,38


Class C parts recorded the highest number of quality incidents and total scrap quantity, followed by Class B and then Class A. However, because the number of parts differs by criticality class, these totals should be normalised before drawing conclusions about relative quality risk.

In [410]:
quality_criticality_rate = parts_by_criticality.merge(
    quality_by_criticality,
    on="criticality_class",
    how="left"
)

quality_criticality_rate["incidents_per_part"] = (
    quality_criticality_rate["incident_count"]
    / quality_criticality_rate["part_count"]
)

quality_criticality_rate["scrap_per_part"] = (
    quality_criticality_rate["total_scrap_qty"]
    / quality_criticality_rate["part_count"]
)

quality_criticality_rate.sort_values(
    "incidents_per_part",
    ascending=False
)

,criticality_class,part_count,incident_count,total_scrap_qty,incidents_per_part,scrap_per_part
1,B,95,139,180,1.463158,1.894737
2,C,157,193,238,1.229299,1.515924
0,A,48,36,38,0.750000,0.791667


After normalising for the number of parts in each criticality class, Class B showed the highest relative quality burden at 1.46 incidents and 1.89 scrap units per part. Class C remained elevated but lower than Class B, while Class A recorded the lowest incident and scrap rates. This suggests quality improvement efforts should prioritise Class B parts rather than relying on raw incident totals alone.

## 7. Key Findings and Recommendations

### Priority 1 — SUP033 Supplier Recovery

**Priority:** High  
**Recommendation:** Launch a supplier recovery plan for SUP033 focused on delivery reliability, corrective actions and contingency sourcing for critical parts.  
**Owner:** Procurement / Supplier Quality  
**Expected Impact:** Improve OTIF, reduce delivery delays and lower exposure on critical parts.  
**Metric to Track:** SUP033 OTIF %, average delivery delay, serious incidents per 100 orders

### Priority 2 — Electrical Inventory Risk

**Priority:** High  
**Recommendation:** Review replenishment and safety-stock settings for Electrical parts, especially P00179, P00124, P00043 and Class A part P00062.  
**Owner:** Supply Chain Planning / Inventory Control  
**Expected Impact:** Reduce recurring stockouts and backorders in the highest-risk part family.  
**Metric to Track:** Stockouts per part, backorders per part, inventory-risk flag count

### Priority 3 — SITE02 Inventory Exposure

**Priority:** High  
**Recommendation:** Investigate SITE02 inventory policy and replenishment performance, with particular focus on Electrical items and Class A part P00062.  
**Owner:** Site Operations / Inventory Control  
**Expected Impact:** Reduce the site’s elevated stockout and backorder exposure and improve availability of critical parts.  
**Metric to Track:** SITE02 stockout count, backorder count, inventory-risk flag count

### Priority 4 — Forecast Accuracy Improvement

**Priority:** Medium  
**Recommendation:** Target forecast improvement work at Cabin parts and SITE06/SITE04, where forecast error is highest. Review demand patterns and forecast assumptions before making wider forecasting changes.  
**Owner:** Demand Planning  
**Expected Impact:** Improve forecast accuracy in the areas with the greatest error while avoiding unnecessary changes in areas where forecasting is already relatively strong.  
**Metric to Track:** MAE by part family, MAE by site

### Priority 5 — Class B Quality Improvement

**Priority:** Medium  
**Recommendation:** Prioritise Class B parts for quality improvement activity, focusing on defect prevention, root-cause analysis and scrap reduction.  
**Owner:** Quality / Engineering  
**Expected Impact:** Reduce the highest normalised quality burden across AeroFlow’s criticality classes and lower avoidable scrap.  
**Metric to Track:** Quality incidents per part, scrap per part, serious incident rate

The analysis indicates that AeroFlow’s main operational risks are not driven by a single root cause. Supplier reliability issues are concentrated around SUP033, inventory availability pressure is strongest within Electrical parts and SITE02, forecast error is concentrated in Cabin parts and selected sites, and Class B parts carry the highest relative quality burden. Improvement actions should therefore be targeted by risk type rather than applied uniformly across the supply chain.